# Data Loading

In [ ]:
!wget https://iasl-btm.iis.sinica.edu.tw/BNER/Content/Revised_JNLPBA.zip

--2025-10-12 18:20:23--  https://iasl-btm.iis.sinica.edu.tw/BNER/Content/Revised_JNLPBA.zip
Resolving iasl-btm.iis.sinica.edu.tw (iasl-btm.iis.sinica.edu.tw)... 140.109.20.133
Connecting to iasl-btm.iis.sinica.edu.tw (iasl-btm.iis.sinica.edu.tw)|140.109.20.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2533317 (2.4M) [application/x-zip-compressed]
Saving to: ‘Revised_JNLPBA.zip.2’

Revised_JNLPBA.zip. 100%[===================>]   2.42M  6.25MB/s    in 0.4s    

2025-10-12 18:20:25 (6.25 MB/s) - ‘Revised_JNLPBA.zip.2’ saved [2533317/2533317]



In [ ]:
!unzip Revised_JNLPBA.zip -d Revised_JNLPBA

Archive:  Revised_JNLPBA.zip
replace Revised_JNLPBA/Genia4EReval1.iob2? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: Revised_JNLPBA/Genia4EReval1.iob2  
replace Revised_JNLPBA/Genia4EReval2.iob2? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: Revised_JNLPBA/Genia4EReval2.iob2  
replace Revised_JNLPBA/Genia4ERtask1.iob2? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: Revised_JNLPBA/Genia4ERtask1.iob2  
replace Revised_JNLPBA/Genia4ERtask2.iob2? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: Revised_JNLPBA/Genia4ERtask2.iob2  


In [ ]:
!pip install seqeval

In [ ]:
import os
import re
from tqdm import tqdm

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification, AutoModel
from sklearn.utils.class_weight import compute_class_weight

from seqeval.metrics import f1_score, classification_report

from google.colab import files

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_NAME = "dmis-lab/biobert-base-cased-v1.1"
MODEL_NAME_LARGE = "dmis-lab/biobert-large-cased-v1.1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer_large = AutoTokenizer.from_pretrained(MODEL_NAME_LARGE)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
def read_iob(file_path):
    sentences, labels = [], []
    tokens, tags = [], []
    header_pattern = re.compile(r"^###MEDLINE:\d+")

    with open(file_path, "r") as f:
      for line in f:
        line = line.strip()
        if not line:  # new sentence
          if tokens:
            sentences.append(tokens)
            labels.append(tags)
            tokens, tags = [], []
          continue


        if header_pattern.match(line):
          continue

        parts = line.split()
        if len(parts) == 2:
          word, tag = parts
          tokens.append(word)
          tags.append(tag)

      # last sentence
      if tokens:
        sentences.append(tokens)
        labels.append(tags)

    return sentences, labels


train_sent1, train_labels1 = read_iob("Revised_JNLPBA/Genia4ERtask1.iob2")
train_sent2, train_labels2 = read_iob("Revised_JNLPBA/Genia4ERtask2.iob2")
eval_sent1, eval_labels1   = read_iob("Revised_JNLPBA/Genia4EReval1.iob2")
eval_sent2, eval_labels2   = read_iob("Revised_JNLPBA/Genia4EReval2.iob2")

train_sentences = train_sent1 + train_sent2
train_labels    = train_labels1 + train_labels2

test_sentences  = eval_sent1 + eval_sent2
test_labels     = eval_labels1 + eval_labels2

train_sentences[0], train_labels[0]

(['IL-2',
  'gene',
  'expression',
  'and',
  'NF-kappa',
  'B',
  'activation',
  'through',
  'CD28',
  'requires',
  'reactive',
  'oxygen',
  'production',
  'by',
  '5-lipoxygenase',
  '.'],
 ['B-DNA',
  'I-DNA',
  'O',
  'O',
  'B-protein',
  'I-protein',
  'O',
  'O',
  'B-protein',
  'O',
  'O',
  'O',
  'O',
  'O',
  'B-protein',
  'O'])

In [ ]:
unique_labels = sorted({lbl for seq in (train_labels+test_labels) for lbl in seq})
label2id = {label: i for i, label in enumerate(unique_labels)}
id2label = {i: label for label, i in label2id.items()}
num_labels = len(unique_labels)

print("Labels:", unique_labels)

Labels: ['B-DNA', 'B-RNA', 'B-cell_line', 'B-cell_type', 'B-protein', 'I-DNA', 'I-RNA', 'I-cell_line', 'I-cell_type', 'I-protein', 'O']


In [ ]:
class NERDataset(Dataset):
  def __init__(self, sentences, labels, tokenizer, label2id, max_len=256):
    self.sentences = sentences
    self.labels = labels
    self.tokenizer = tokenizer
    self.label2id = label2id
    self.max_len = max_len

  def __len__(self):
    return len(self.sentences)

  def __getitem__(self, idx):
    # Tokenization
    tokens = self.sentences[idx]
    labels = self.labels[idx]

    encoding = self.tokenizer(
      tokens,
      is_split_into_words=True,
      truncation=True,
      padding='max_length',
      max_length=self.max_len,
      return_tensors='pt'
    )

    # Label Aligning for bert
    word_ids = encoding.word_ids(batch_index=0)

    label_ids = []
    for w_id in word_ids:
      if w_id is None:
        label_ids.append(-100)
      else:
        label_ids.append(self.label2id[labels[w_id]])

    label_ids = label_ids[:self.max_len] + [-100] * (self.max_len - len(label_ids))
    item = {key: val.squeeze(0) for key, val in encoding.items()}
    item["labels"] = torch.tensor(label_ids)
    return item

#Biobert Base

In [ ]:
train_dataset = NERDataset(train_sentences, train_labels, tokenizer, label2id)
test_dataset = NERDataset(test_sentences, test_labels, tokenizer, label2id)

## 32 bs, adamw 5e-5 1e-12, 3 epochs

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

batch = next(iter(train_loader))
print(batch['input_ids'].shape)
print(batch['labels'].shape)

torch.Size([32, 256])
torch.Size([32, 256])


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, eps=1e-12)

epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}")

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1/3:   0%|          | 0/1160 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Epoch 1/3: 100%|██████████| 1160/1160 [11:31<00:00,  1.68it/s]


Epoch 1: loss=0.1366


Epoch 2/3: 100%|██████████| 1160/1160 [11:36<00:00,  1.66it/s]


Epoch 2: loss=0.0456


Epoch 3/3: 100%|██████████| 1160/1160 [11:37<00:00,  1.66it/s]

Epoch 3: loss=0.0272


In [ ]:
save_dir = "/content/biobert_ner_model_3epoch"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Model saved to {save_dir}")

Model saved to /content/biobert_ner_model_3epoch


In [ ]:
!zip -r biobert_ner_model_3epoch.zip /content/biobert_ner_model_3epoch
files.download("biobert_ner_model_3epoch.zip")

updating: content/biobert_ner_model_3epoch/ (stored 0%)
updating: content/biobert_ner_model_3epoch/tokenizer.json (deflated 70%)
updating: content/biobert_ner_model_3epoch/vocab.txt (deflated 49%)
updating: content/biobert_ner_model_3epoch/special_tokens_map.json (deflated 42%)
updating: content/biobert_ner_model_3epoch/tokenizer_config.json (deflated 74%)
updating: content/biobert_ner_model_3epoch/config.json (deflated 57%)
updating: content/biobert_ner_model_3epoch/model.safetensors (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model.eval()
true_labels, pred_labels = [], []

with torch.inference_mode():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=2)

        for i in range(len(labels)):
            true_seq, pred_seq = [], []
            for j in range(len(labels[i])):
                if labels[i][j] == -100:
                    continue
                true_seq.append(id2label[labels[i][j].item()])
                pred_seq.append(id2label[predictions[i][j].item()])
            true_labels.append(true_seq)
            pred_labels.append(pred_seq)

print(classification_report(true_labels, pred_labels))
print("F1:", f1_score(true_labels, pred_labels))


              precision    recall  f1-score   support

         DNA       0.86      0.90      0.88      5736
         RNA       0.84      0.76      0.80      1188
   cell_line       0.93      0.90      0.92      2614
   cell_type       0.89      0.90      0.89      9832
     protein       0.92      0.89      0.91     37236

   micro avg       0.90      0.89      0.90     56606
   macro avg       0.89      0.87      0.88     56606
weighted avg       0.91      0.89      0.90     56606

F1: 0.8989367376693573


## 32 bs, adamw 5e-5 1e-12, 5 epochs

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, eps=1e-12)

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1/5: 100%|██████████| 1160/1160 [11:35<00:00,  1.67it/s]


Epoch 1: loss=0.1324


Epoch 2/5: 100%|██████████| 1160/1160 [11:35<00:00,  1.67it/s]


Epoch 2: loss=0.0438


Epoch 3/5: 100%|██████████| 1160/1160 [11:35<00:00,  1.67it/s]


Epoch 3: loss=0.0256


Epoch 4/5: 100%|██████████| 1160/1160 [11:35<00:00,  1.67it/s]


Epoch 4: loss=0.0173


Epoch 5/5: 100%|██████████| 1160/1160 [11:35<00:00,  1.67it/s]

Epoch 5: loss=0.0144


In [ ]:
save_dir = "/content/biobert_ner_model_5epoch"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Model saved to {save_dir}")

!zip -r biobert_ner_model_5epoch.zip /content/biobert_ner_model_5epoch
files.download("biobert_ner_model_5epoch.zip")

Model saved to /content/biobert_ner_model_5epoch
  adding: content/biobert_ner_model_5epoch/ (stored 0%)
  adding: content/biobert_ner_model_5epoch/tokenizer.json (deflated 70%)
  adding: content/biobert_ner_model_5epoch/vocab.txt (deflated 49%)
  adding: content/biobert_ner_model_5epoch/special_tokens_map.json (deflated 42%)
  adding: content/biobert_ner_model_5epoch/tokenizer_config.json (deflated 74%)
  adding: content/biobert_ner_model_5epoch/config.json (deflated 57%)
  adding: content/biobert_ner_model_5epoch/model.safetensors (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model.eval()
true_labels, pred_labels = [], []

with torch.inference_mode():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=2)

        for i in range(len(labels)):
            true_seq, pred_seq = [], []
            for j in range(len(labels[i])):
                if labels[i][j] == -100:
                    continue
                true_seq.append(id2label[labels[i][j].item()])
                pred_seq.append(id2label[predictions[i][j].item()])
            true_labels.append(true_seq)
            pred_labels.append(pred_seq)

print(classification_report(true_labels, pred_labels))
print("F1:", f1_score(true_labels, pred_labels))

              precision    recall  f1-score   support

         DNA       0.85      0.90      0.88      5736
         RNA       0.75      0.77      0.76      1188
   cell_line       0.90      0.94      0.92      2614
   cell_type       0.91      0.89      0.90      9832
     protein       0.92      0.89      0.90     37236

   micro avg       0.91      0.89      0.90     56606
   macro avg       0.87      0.88      0.87     56606
weighted avg       0.91      0.89      0.90     56606

F1: 0.8980494240987055


## 16 bs, adamw 2e-8 1e-8, 10 epochs




In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

batch = next(iter(train_loader))
print(batch['input_ids'].shape)
print(batch['labels'].shape)

torch.Size([16, 256])
torch.Size([16, 256])


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-8, eps=1e-8)

epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-base-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1/10: 100%|██████████| 2319/2319 [11:59<00:00,  3.22it/s]


Epoch 1: loss=2.3460


Epoch 2/10: 100%|██████████| 2319/2319 [11:59<00:00,  3.22it/s]


Epoch 2: loss=2.0166


Epoch 3/10: 100%|██████████| 2319/2319 [11:59<00:00,  3.22it/s]


...

Epoch 8: loss=0.8820


Epoch 9/10: 100%|██████████| 2319/2319 [11:59<00:00,  3.22it/s]


Epoch 9: loss=0.8246


Epoch 10/10: 100%|██████████| 2319/2319 [11:59<00:00,  3.22it/s]

Epoch 10: loss=0.7767


In [ ]:
save_dir = "/content/biobert_ner_model_10epoch"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Model saved to {save_dir}")

!zip -r biobert_ner_model_5epoch.zip /content/biobert_ner_model_5epoch
files.download("biobert_ner_model_10epoch.zip")

Model saved to /content/biobert_ner_model_5epoch
updating: content/biobert_ner_model_5epoch/ (stored 0%)
updating: content/biobert_ner_model_5epoch/tokenizer.json (deflated 70%)
updating: content/biobert_ner_model_5epoch/vocab.txt (deflated 49%)
updating: content/biobert_ner_model_5epoch/special_tokens_map.json (deflated 42%)
updating: content/biobert_ner_model_5epoch/tokenizer_config.json (deflated 74%)
updating: content/biobert_ner_model_5epoch/config.json (deflated 57%)
updating: content/biobert_ner_model_5epoch/model.safetensors (deflated 7%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model.eval()
true_labels, pred_labels = [], []

with torch.inference_mode():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=2)

        for i in range(len(labels)):
            true_seq, pred_seq = [], []
            for j in range(len(labels[i])):
                if labels[i][j] == -100:
                    continue
                true_seq.append(id2label[labels[i][j].item()])
                pred_seq.append(id2label[predictions[i][j].item()])
            true_labels.append(true_seq)
            pred_labels.append(pred_seq)

print(classification_report(true_labels, pred_labels))
print("F1:", f1_score(true_labels, pred_labels))

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

         DNA       0.00      0.00      0.00      5736
         RNA       0.00      0.00      0.00      1188
   cell_line       0.00      0.00      0.00      2614
   cell_type       0.25      0.00      0.00      9832
     protein       0.51      0.67      0.57     37236

   micro avg       0.51      0.44      0.47     56606
   macro avg       0.15      0.13      0.12     56606
weighted avg       0.38      0.44      0.38     56606

F1: 0.4693761599939396


# Biobert Large

In [ ]:
train_dataset = NERDataset(train_sentences, train_labels, tokenizer, label2id)
test_dataset = NERDataset(test_sentences, test_labels, tokenizer, label2id)

## 16 bs, adam2 1e-5 1e-12, 3 epochs

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16)

batch = next(iter(train_loader))
print(batch['input_ids'].shape)
print(batch['labels'].shape)

torch.Size([16, 256])
torch.Size([16, 256])


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME_LARGE, num_labels=num_labels, id2label=id2label, label2id=label2id
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, eps=1e-12)

epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}")

pytorch_model.bin:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-large-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.

Epoch 1/3: 100%|██████████| 2319/2319 [37:12<00:00,  1.04it/s]


Epoch 1: loss=0.1765


Epoch 2/3: 100%|██████████| 2319/2319 [36:48<00:00,  1.05it/s]


Epoch 2: loss=0.0643


Epoch 3/3: 100%|██████████| 2319/2319 [36:48<00:00,  1.05it/s]

Epoch 3: loss=0.0371


In [ ]:
save_dir = "/content/biobert_LARGE_model_3epoch"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Model saved to {save_dir}")

!zip -r biobert_LARGE_model_3epoch.zip /content/biobert_LARGE_model_3epoch
files.download("biobert_LARGE_model_3epoch.zip")

Model saved to /content/biobert_LARGE_model_3epoch
  adding: content/biobert_LARGE_model_3epoch/ (stored 0%)
  adding: content/biobert_LARGE_model_3epoch/tokenizer_config.json (deflated 74%)
  adding: content/biobert_LARGE_model_3epoch/vocab.txt (deflated 49%)
  adding: content/biobert_LARGE_model_3epoch/special_tokens_map.json (deflated 42%)
  adding: content/biobert_LARGE_model_3epoch/tokenizer.json (deflated 70%)
  adding: content/biobert_LARGE_model_3epoch/config.json (deflated 57%)
  adding: content/biobert_LARGE_model_3epoch/model.safetensors (deflated 11%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model.eval()
true_labels, pred_labels = [], []

with torch.inference_mode():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=2)

        for i in range(len(labels)):
            true_seq, pred_seq = [], []
            for j in range(len(labels[i])):
                if labels[i][j] == -100:
                    continue
                true_seq.append(id2label[labels[i][j].item()])
                pred_seq.append(id2label[predictions[i][j].item()])
            true_labels.append(true_seq)
            pred_labels.append(pred_seq)

print(classification_report(true_labels, pred_labels))
print("F1:", f1_score(true_labels, pred_labels))

              precision    recall  f1-score   support

         DNA       0.80      0.89      0.84      5736
         RNA       0.91      0.72      0.80      1188
   cell_line       0.88      0.93      0.91      2614
   cell_type       0.88      0.90      0.89      9832
     protein       0.93      0.86      0.89     37236

   micro avg       0.90      0.87      0.89     56606
   macro avg       0.88      0.86      0.87     56606
weighted avg       0.91      0.87      0.89     56606

F1: 0.885844913240694


##16 bs, adamw 1e-5 1e-12, 5 epochs

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME_LARGE, num_labels=num_labels, id2label=id2label, label2id=label2id
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5, eps=1e-12)

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-large-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1/5: 100%|██████████| 2319/2319 [36:47<00:00,  1.05it/s]


Epoch 1: loss=0.1548


Epoch 2/5: 100%|██████████| 2319/2319 [36:48<00:00,  1.05it/s]


Epoch 2: loss=0.0538


Epoch 3/5: 100%|██████████| 2319/2319 [36:47<00:00,  1.05it/s]


Epoch 3: loss=0.0311


Epoch 4/5: 100%|██████████| 2319/2319 [36:48<00:00,  1.05it/s]


Epoch 4: loss=0.0193


Epoch 5/5: 100%|██████████| 2319/2319 [36:48<00:00,  1.05it/s]

Epoch 5: loss=0.0133


In [ ]:
save_dir = "/content/biobert_LARGE_model_5epoch"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Model saved to {save_dir}")

!zip -r biobert_LARGE_model_5epoch.zip /content/biobert_LARGE_model_5epoch
files.download("biobert_LARGE_model_5epoch.zip")

Model saved to /content/biobert_LARGE_model_5epoch
  adding: content/biobert_LARGE_model_5epoch/ (stored 0%)
  adding: content/biobert_LARGE_model_5epoch/tokenizer_config.json (deflated 74%)
  adding: content/biobert_LARGE_model_5epoch/vocab.txt (deflated 49%)
  adding: content/biobert_LARGE_model_5epoch/special_tokens_map.json (deflated 42%)
  adding: content/biobert_LARGE_model_5epoch/tokenizer.json (deflated 70%)
  adding: content/biobert_LARGE_model_5epoch/config.json (deflated 57%)
  adding: content/biobert_LARGE_model_5epoch/model.safetensors (deflated 11%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model.eval()
true_labels, pred_labels = [], []

with torch.inference_mode():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=2)

        for i in range(len(labels)):
            true_seq, pred_seq = [], []
            for j in range(len(labels[i])):
                if labels[i][j] == -100:
                    continue
                true_seq.append(id2label[labels[i][j].item()])
                pred_seq.append(id2label[predictions[i][j].item()])
            true_labels.append(true_seq)
            pred_labels.append(pred_seq)

print(classification_report(true_labels, pred_labels))
print("F1:", f1_score(true_labels, pred_labels))

              precision    recall  f1-score   support

         DNA       0.88      0.92      0.90      5736
         RNA       0.95      0.70      0.80      1188
   cell_line       0.89      0.93      0.91      2614
   cell_type       0.89      0.89      0.89      9832
     protein       0.92      0.88      0.90     37236

   micro avg       0.91      0.88      0.90     56606
   macro avg       0.91      0.86      0.88     56606
weighted avg       0.91      0.88      0.90     56606

F1: 0.8974735710446157


## 16 bs, adamw 5e-5 1e-12 3 epochs

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME_LARGE, num_labels=num_labels, id2label=id2label, label2id=label2id
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, eps=1e-12)

epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}")

Some weights of BertForTokenClassification were not initialized from the model checkpoint at dmis-lab/biobert-large-cased-v1.1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1/3: 100%|██████████| 2319/2319 [36:48<00:00,  1.05it/s]


Epoch 1: loss=0.1632


Epoch 2/3: 100%|██████████| 2319/2319 [36:48<00:00,  1.05it/s]


Epoch 2: loss=0.0755


Epoch 3/3: 100%|██████████| 2319/2319 [36:48<00:00,  1.05it/s]

Epoch 3: loss=0.0661


In [ ]:
save_dir = "/content/biobert_LARGE_model_5epoch"

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print(f"Model saved to {save_dir}")

!zip -r biobert_LARGE_model_5epoch.zip /content/biobert_LARGE_model_5epoch
files.download("biobert_LARGE_model_5epoch.zip")

Model saved to /content/biobert_LARGE_model_5epoch
updating: content/biobert_LARGE_model_5epoch/ (stored 0%)
updating: content/biobert_LARGE_model_5epoch/tokenizer_config.json (deflated 74%)
updating: content/biobert_LARGE_model_5epoch/vocab.txt (deflated 49%)
updating: content/biobert_LARGE_model_5epoch/special_tokens_map.json (deflated 42%)
updating: content/biobert_LARGE_model_5epoch/tokenizer.json (deflated 70%)
updating: content/biobert_LARGE_model_5epoch/config.json (deflated 57%)
updating: content/biobert_LARGE_model_5epoch/model.safetensors (deflated 11%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
model.eval()
true_labels, pred_labels = [], []

with torch.inference_mode():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs.logits, dim=2)

        for i in range(len(labels)):
            true_seq, pred_seq = [], []
            for j in range(len(labels[i])):
                if labels[i][j] == -100:
                    continue
                true_seq.append(id2label[labels[i][j].item()])
                pred_seq.append(id2label[predictions[i][j].item()])
            true_labels.append(true_seq)
            pred_labels.append(pred_seq)

print(classification_report(true_labels, pred_labels))
print("F1:", f1_score(true_labels, pred_labels))

              precision    recall  f1-score   support

         DNA       0.86      0.86      0.86      5736
         RNA       0.83      0.76      0.79      1188
   cell_line       0.91      0.85      0.88      2614
   cell_type       0.89      0.87      0.88      9832
     protein       0.89      0.89      0.89     37236

   micro avg       0.89      0.88      0.88     56606
   macro avg       0.88      0.85      0.86     56606
weighted avg       0.89      0.88      0.88     56606

F1: 0.8843802604766545


# Best Model + Weighting

In [ ]:
class BIOBERT_Weighted(nn.Module):
    def __init__(self, model_name, num_labels, class_weights=None):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

        if class_weights is not None:
            self.register_buffer("class_weights", torch.tensor(class_weights, dtype=torch.float))
        else:
            self.class_weights = None

    def forward(self, input_ids, attention_mask=None, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs.last_hidden_state)
        logits = self.classifier(sequence_output)

        if labels is not None:
            if self.class_weights is not None:
                loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device), ignore_index=-100)
            else:
                loss_fct = nn.CrossEntropyLoss(ignore_index=-100)

            loss = loss_fct(logits.view(-1, logits.shape[-1]), labels.view(-1))
            return {"loss": loss, "logits": logits}
        else:
            return {"logits": logits}

In [ ]:
train_dataset = NERDataset(train_sentences, train_labels, tokenizer, label2id)
test_dataset = NERDataset(test_sentences, test_labels, tokenizer, label2id)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

batch = next(iter(train_loader))
print(batch['input_ids'].shape)
print(batch['labels'].shape)

torch.Size([32, 256])
torch.Size([32, 256])


In [ ]:
labels_flat = [label2id[l] for seq in (train_labels + test_labels) for l in seq if l in label2id]
unique_labels = np.unique(labels_flat)
class_weights = compute_class_weight('balanced', classes=unique_labels, y=labels_flat)
tensor_weights = torch.tensor(class_weights, dtype=torch.float)

In [ ]:
model = BIOBERT_Weighted(MODEL_NAME, num_labels=len(label2id), class_weights=tensor_weights)
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, eps=1e-12)

epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in loop:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs['loss']
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}")

/tmp/ipython-input-2244878403.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer("class_weights", torch.tensor(class_weights, dtype=torch.float))
Epoch 1/3: 100%|██████████| 1160/1160 [11:28<00:00,  1.69it/s]


Epoch 1: loss=0.2530


Epoch 2/3: 100%|██████████| 1160/1160 [11:29<00:00,  1.68it/s]


Epoch 2: loss=0.0896


Epoch 3/3: 100%|██████████| 1160/1160 [11:28<00:00,  1.68it/s]

Epoch 3: loss=0.0652


In [ ]:
model.eval()
true_labels, pred_labels = [], []

with torch.inference_mode():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        predictions = torch.argmax(outputs['logits'], dim=2)

        for i in range(len(labels)):
            true_seq, pred_seq = [], []
            for j in range(len(labels[i])):
                if labels[i][j] == -100:
                    continue
                true_seq.append(id2label[labels[i][j].item()])
                pred_seq.append(id2label[predictions[i][j].item()])
            true_labels.append(true_seq)
            pred_labels.append(pred_seq)

print(classification_report(true_labels, pred_labels))
print("F1:", f1_score(true_labels, pred_labels))

              precision    recall  f1-score   support

         DNA       0.71      0.93      0.81      5736
         RNA       0.56      0.89      0.68      1188
   cell_line       0.53      0.95      0.68      2614
   cell_type       0.79      0.92      0.85      9832
     protein       0.88      0.90      0.89     37236

   micro avg       0.81      0.91      0.86     56606
   macro avg       0.69      0.92      0.78     56606
weighted avg       0.82      0.91      0.86     56606

F1: 0.8562705339660838


In [ ]:
save_path = "/content/biobert_weighted.pt"

torch.save(model.state_dict(), save_path)
files.download("biobert_weighted.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>